# Lecture: Exchange-Correlation Functionals in Density Functional Theory

## 1. Introduction: from the electron to the everyday life

Before diving into the mathematics of functionals, consider that virtually everything in our daily lives—the air we breathe, the materials around us, the technologies we rely on, even life itself—results from the interaction of many electrons through the electromagnetic force. Physics binds particles into atoms, chemistry emerges from how those atoms bond, and biology builds upon that chemical tapestry. This is **many-body physics**: the collective behavior of vast numbers of interacting particles, governed by quantum mechanics. In principle, the **Schrödinger equation** describes it all exactly; in practice, solving it for any system of real-world interest is impossibly difficult.

**Density Functional Theory (DFT)** offers a brilliant escape from this exponential complexity. Instead of grappling with an unmanageable many-electron wave function, DFT reframes the problem in terms of the simple, three-dimensional **electron density**, $n(\mathbf{r})$. The Kohn-Sham approach maps the real interacting system onto a fictitious system of non-interacting particles that share the same density, bundling all the complex quantum mechanical effects—exchange (Pauli exclusion) and correlation (Coulomb repulsion)—into a single term: the **exchange-correlation functional**, $E_{\text{xc}}[n]$. If we knew its exact form, DFT would be exact. We do not, and approximating this one term is where all the art, science, and history of modern DFT resides. This lecture explores what goes into $E_{\text{xc}}[n]$, why it is so powerful, and how different approximations along "Jacob's Ladder" transform the hopeless complexity of the many-electron problem into the predictive computational tool we rely on today.

## 2. Lecture Overview & Learning Objectives
This lecture introduces the central concept of exchange-correlation in DFT. By the end of this session, students will be able to:
- Explain why exchange-correlation functionals are necessary in the Kohn-Sham approach
- Differentiate between major functional classes (LDA, GGA, meta-GGA, hybrid)
- Understand the strengths and limitations of each approximation
- Make informed choices about functional selection for specific systems

## 3. The Many-Electron Problem

We begin with the full, time-independent, non-relativistic Schrödinger equation for a system of $N$ interacting electrons moving in the electrostatic potential of fixed nuclei:

$$\hat{H}\Psi(\mathbf{r}_1, \mathbf{r}_2, ..., \mathbf{r}_N) = E\Psi(\mathbf{r}_1, \mathbf{r}_2, ..., \mathbf{r}_N)$$

The Hamiltonian contains the kinetic energy of each electron, the attractive potential from the nuclei $V_{\text{ext}}(\mathbf{r})$, and the repulsive interactions between every pair of electrons:

$$\hat{H} = \sum_{i=1}^N \left[ -\frac{\hbar^2}{2m_e}\nabla_i^2 + V_{\text{ext}}(\mathbf{r}_i) \right] + \frac{1}{2}\sum_{i \neq j}^N \frac{e^2}{|\mathbf{r}_i - \mathbf{r}_j|}$$

The wave function $\Psi$ is a function of $3N$ coordinates, and solving this equation directly for any system with more than a handful of electrons is exponentially impossible. To illustrate the sheer impossibility of a direct attack, consider a small molecule with just **10 electrons**. Its wave function is a function of 30 coordinates (3 per electron). To represent this function with even a crude grid of just 10 points per dimension, you would need $10^{30}$ numbers. Storing that many numbers, with even a single byte each, would require more data than all the digital information ever created by humankind. For a protein with thousands of electrons, the problem is not just difficult; it is **exponentially impossible**.

## 4. Density functional theory

**Density Functional Theory** provides an alternative formulation. The Hohenberg-Kohn theorems establish a one-to-one mapping between the ground state wave function and the ground state **electron density**, $n(\mathbf{r})$. This allows us, in principle, to express the total energy as a functional of the density alone:

$$E[n] = T[n] + \int V_{\text{ext}}(\mathbf{r}) n(\mathbf{r}) d\mathbf{r} + \frac{1}{2}\int\int \frac{n(\mathbf{r})n(\mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|} d\mathbf{r}d\mathbf{r}' + E'_{\text{xc}}[n]$$

Here, $T[n]$ is the kinetic energy of the *interacting* electron system—a quantity that is not easily expressed directly in terms of the density.


The total energy functional $E[n]$ consists of four distinct contributions. The **kinetic energy term $T[n]$** represents the quantum mechanical motion of the electrons, though in practice this is difficult to express directly in terms of the density and is instead handled via the Kohn-Sham orbitals (we will explain them next). The **external potential term $\int V_{\text{ext}}(\mathbf{r}) n(\mathbf{r}) d\mathbf{r}$** captures the interaction between the electrons and the nuclei, where $V_{\text{ext}}(\mathbf{r})$ is the attractive potential generated by the fixed atomic nuclei. The **Hartree term $\frac{1}{2}\int\int \frac{n(\mathbf{r})n(\mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|} d\mathbf{r}d\mathbf{r}'$** describes the classical electrostatic repulsion between electrons, treating the electron density as a continuous charge distribution interacting with itself. Finally, the **exchange-correlation term $E'_{\text{xc}}[n]$** contains everything else: the quantum mechanical exchange energy arising from the Pauli exclusion principle, the correlation energy from instantaneous electron-electron interactions.


### 4.1 Kohn-Sham equation: Mapping a many-body problem onto a non-interacting one

The **Kohn-Sham approach** is to introduce a fictitious system of $N$ *non-interacting* electrons that have the **same ground state density** $n(\mathbf{r})$ as the real, interacting system. This mapping transforms the intractable many-body problem into a set of solvable single-particle equations, known as the **Kohn-Sham equations**:

$$\left[-\frac{\hbar^2}{2m_e}\nabla^2 + V_{\text{KS}}(\mathbf{r})\right] \psi_i(\mathbf{r}) = \epsilon_i \psi_i(\mathbf{r})$$

Here, $\psi_i(\mathbf{r})$ are the Kohn-Sham orbitals, $\epsilon_i$ are the corresponding eigenvalues, and $V_{\text{KS}}(\mathbf{r})$ is an effective potential experienced by each non-interacting electron. This potential is given by:

$$V_{\text{KS}}(\mathbf{r}) = V_{\text{ext}}(\mathbf{r}) + V_{\text{Hartree}}(\mathbf{r}) + V_{\text{xc}}(\mathbf{r})$$

Where:
- $V_{\text{ext}}(\mathbf{r})$ is the external potential from the nuclei.
- $V_{\text{Hartree}}(\mathbf{r}) = \int \frac{n(\mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|} d\mathbf{r}'$ is the classical electrostatic potential due to the total electron density.
- $V_{\text{xc}}(\mathbf{r}) = \frac{\delta E_{\text{xc}}[n]}{\delta n(\mathbf{r})}$ is the **exchange-correlation potential**. This term contains **all quantum mechanical effects** not captured by the classical Hartree potential—most notably exchange (due to the Pauli principle) and correlation (due to instantaneous electron-electron interactions).

The Kohn-Sham equations have the form of a simple Schrödinger equation for non-interacting particles moving in an effective potential. This is the central achievement of the method: it replaces the impossible task of solving a single, giant 3N-dimensional equation with the manageable task of solving $N$ coupled three-dimensional equations. The coupling occurs through the density, which is constructed from the orbitals themselves:

$$n(\mathbf{r}) = \sum_{i=1}^N f_i |\psi_i(\mathbf{r})|^2$$

where $f_i$ are orbital occupations. Because the Kohn-Sham potential $V_{\text{KS}}(\mathbf{r})$ depends on the density, which depends on the orbitals, these equations must be solved **self-consistently**. The exchange-correlation term, $V_{\text{xc}}(\mathbf{r})$, is the only part of the Kohn-Sham potential that is not known exactly; all other terms are computed directly from the density or the external potential. Thus, the accuracy of any DFT calculation rests entirely on the quality of the approximation used for $E_{\text{xc}}[n]$ and its corresponding potential.




The kinetic energy of the non-interacting Kohn-Sham system, denoted $T_s[n]$, is evaluated explicitly from the Kohn-Sham orbitals $\psi_i(\mathbf{r})$. Since these orbitals form a Slater determinant describing the fictitious non-interacting system, the kinetic energy is simply the sum of the single-particle kinetic energies:

$$T_s[n] = \sum_{i=1}^N \int \psi_i^*(\mathbf{r}) \left( -\frac{\hbar^2}{2m_e}\nabla^2 \right) \psi_i(\mathbf{r}) d\mathbf{r} = \sum_{i=1}^N \langle \psi_i | \hat{T} | \psi_i \rangle$$

This expression is exact for the non-interacting system that reproduces the true density $n(\mathbf{r})$, but it differs from the kinetic energy $T[n]$ of the fully interacting system. The difference between these two kinetic energies is absorbed into the exchange-correlation functional $E_{\text{xc}}[n]$, which is why $E_{\text{xc}}[n]$ contains not only quantum mechanical exchange and correlation effects but also a kinetic energy correction term.

We then write the total energy as:

$$E[n] = T_s[n] + \int V_{\text{ext}}(\mathbf{r}) n(\mathbf{r}) d\mathbf{r} + \frac{1}{2}\int\int \frac{n(\mathbf{r})n(\mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|} d\mathbf{r}d\mathbf{r}' + E_{\text{xc}}[n]$$

By construction, we have moved all the difficult many-body physics—the difference between the true kinetic energy $T[n]$ and the non-interacting kinetic energy $T_s[n]$, as well as all the complexities of electron-electron exchange and correlation beyond the classical Hartree term—into a single, mysterious quantity: **the exchange-correlation functional, $E_{\text{xc}}[n]$** .

Formally, $E_{\text{xc}}[n]$ is defined by the equation above. It contains:
- **Exchange energy:** Arising from the antisymmetry of the wave function (Pauli exclusion principle), which keeps electrons of like spin apart.
- **Correlation energy:** Arising from the instantaneous Coulomb repulsion between electrons, which causes their motions to be correlated.
- **A kinetic energy correction:** The difference between the kinetic energy of the true interacting system and the fictitious non-interacting system.

If the exact form of $E_{\text{xc}}[n]$ were known, the Kohn-Sham approach would yield the exact ground state energy and density. Its existence is proven, but its exact form is not known.


The genius of this approach is twofold: first, it replaces the impossible task of solving a single 3N-dimensional Schrödinger equation with the tractable problem of solving $N$ coupled three-dimensional equations; second, it strategically isolates the computationally difficult many-body effects. By explicitly computing the independent-particle kinetic energy $T_s[n]$ and the long-range Hartree term exactly from the orbitals and density, the Kohn-Sham scheme ensures that only the remaining exchange-correlation functional $E_{\text{xc}}[n]$—a relatively small but crucial component of the total energy—must be approximated. This allows $E_{\text{xc}}[n]$ to be reasonably modeled as a local or semi-local functional of the density, forming the foundation of practical DFT calculations.


# 5. Approximation to the exchange-correlation functional

Choosing a functional that fits your system is the critical decision point for performance . The exchange-correlation functional is the only approximated term in Kohn-Sham DFT—all other terms are exact.
Next we discuss the most common approximations to the exchange-correlation functional.

## 5.1 Local Density Approximation (LDA)

The Local Density Approximation (LDA) represents the simplest and historically the first practical approximation for the exchange-correlation functional. It was pioneered by Kohn and Sham in their seminal 1965 paper and remains conceptually important as the foundation upon which all more advanced functionals are built.

The fundamental assumption of LDA is deceptively simple: **at each point in space, the exchange-correlation energy per electron depends only on the local electron density at that point**. In other words, the system is treated locally as if it were a uniform electron gas (UEG) with the same density.

Mathematically, this is expressed as:

$$E_{\text{xc}}^{\text{LDA}}[n] = \int n(\mathbf{r}) \, \varepsilon_{\text{xc}}^{\text{unif}}(n(\mathbf{r})) \, d\mathbf{r}$$

where $\varepsilon_{\text{xc}}^{\text{unif}}(n)$ is the exchange-correlation **energy per electron** of a uniform electron gas of density $n$.

This approximation is exact in the limit of a perfectly homogeneous system (like a simple metal) and might seem crude for real materials where density varies rapidly, particularly near nuclei and between atoms. Yet, remarkably, it works surprisingly well for many systems.

### 5.1.1 Building LDA: The Uniform Electron Gas

To construct an LDA functional, we need $\varepsilon_{\text{xc}}^{\text{unif}}(n)$, which is typically split into exchange and correlation contributions:

$$\varepsilon_{\text{xc}}^{\text{unif}}(n) = \varepsilon_{\text{x}}^{\text{unif}}(n) + \varepsilon_{\text{c}}^{\text{unif}}(n)$$

**Exchange energy:** The exchange part for the uniform electron gas has a simple analytic form, derived by Bloch and Dirac in the late 1920s:

$$\varepsilon_{\text{x}}^{\text{unif}}(n) = -\frac{3}{4}\left(\frac{3}{\pi}\right)^{1/3} n^{1/3}$$

This is often written in terms of the Wigner-Seitz radius $r_s = (3/4\pi n)^{1/3}$ as $\varepsilon_{\text{x}}^{\text{unif}} = -3/(4\pi r_s)$ in atomic units.

**Correlation energy:** The correlation part is more complicated and has no simple exact analytic form. It was accurately determined for a range of densities by **Ceperley and Alder in 1980** using quantum Monte Carlo simulations of the uniform electron gas. Various analytical parameterizations of these numerical results were subsequently developed, with the most widely used being:

- **Perdew-Zunger (1981):** A parameterization based on fitting the Ceperley-Alder data
- **Vosko-Wilk-Nusair (VWN, 1980):** Another popular parameterization developed around the same time
- **Perdew-Wang (PW92, 1992):** A more modern and accurate parameterization

These parameterizations provide smooth, differentiable expressions for $\varepsilon_{\text{c}}^{\text{unif}}(r_s)$ that can be used in practical calculations.

### 5.1.2 The LDA Exchange-Correlation Potential

The corresponding exchange-correlation potential, required for the Kohn-Sham equations, is obtained by taking the functional derivative:

$$V_{\text{xc}}^{\text{LDA}}(\mathbf{r}) = \frac{\delta E_{\text{xc}}^{\text{LDA}}[n]}{\delta n(\mathbf{r})} = \varepsilon_{\text{xc}}^{\text{unif}}(n(\mathbf{r})) + n(\mathbf{r}) \frac{d\varepsilon_{\text{xc}}^{\text{unif}}(n)}{dn}\bigg|_{n=n(\mathbf{r})}$$

This potential depends **only** on the density at point $\mathbf{r}$ and its local derivative with respect to density, making it computationally very efficient.

### Strengths and Successes

Despite its simplicity, LDA achieves remarkable success for certain classes of systems:

- **Fast computation:** LDA is the most computationally efficient class of functionals, scaling linearly with system size
- **Accurate for metals:** Many simple metals are well-described by LDA because their electron density resembles the uniform electron gas
- **Good crystal structures:** LDA often predicts lattice constants with surprising accuracy, typically within 1-2% of experiment (though it tends to overbind, giving slightly too short bonds)
- **Reliable for bulk moduli and vibrational frequencies:** These properties are often well-reproduced
- **Exact constraints:** LDA satisfies several important exact constraints, including the sum rule for the exchange-correlation hole

### Limitations and Failures

LDA's approximations break down in important ways, revealing where more sophisticated functionals are needed:

- **Overbinding and bond lengths:** LDA tends to overestimate binding energies and underestimate bond lengths (typically by ~1-2%)
- **Band gap problem:** Semiconductor and insulator band gaps are severely underestimated, often by 30-50% or more, sometimes predicting metals where insulators exist
- **No van der Waals interactions:** LDA completely fails to capture long-range dispersion forces, which are essential for molecular crystals, layered materials (like graphite), and biological systems
- **Self-interaction error:** An electron interacting with itself through the Hartree term is not fully canceled by LDA exchange-correlation, leading to overly delocalized densities and incorrect dissociation limits
- **Poor for hydrogen bonding:** LDA describes hydrogen bonds poorly, limiting its use in biological and aqueous systems
- **Reaction barriers:** Transition state energies and reaction barriers are typically too low

### When to Use LDA

LDA is rarely the functional of choice for high-precision work today, but it remains useful in specific contexts:

- **Large-scale screening studies** where computational cost is the primary constraint
- **Metals and simple solids** where its errors are often systematic and well-understood
- **As a starting point** for more accurate calculations (e.g., providing initial densities for hybrid functionals)
- **Pedagogically**, as the foundation upon which all modern functional development is built

### Spin-Polarized Systems: LSDA

For magnetic systems or open-shell molecules, LDA is extended to the **Local Spin Density Approximation (LSDA)** . Here, the functional depends separately on the spin-up density $n_\uparrow(\mathbf{r})$ and spin-down density $n_\downarrow(\mathbf{r})$:

$$E_{\text{xc}}^{\text{LSDA}}[n_\uparrow, n_\downarrow] = \int n(\mathbf{r}) \, \varepsilon_{\text{xc}}^{\text{unif}}(n_\uparrow(\mathbf{r}), n_\downarrow(\mathbf{r})) \, d\mathbf{r}$$

This allows LSDA to describe ferromagnetic materials (like iron and nickel) and spin-polarized atoms and molecules. The exchange energy for a spin-polarized uniform gas has an exact form (the Dirac exchange generalized by von Barth and Hedin), and correlation parameterizations exist for arbitrary spin polarization.


## 5.2 Generalized Gradient Approximation (GGA)

The Local Density Approximation treats each point in space as if it were part of a uniform electron gas, completely ignoring any information about how the density varies in the surrounding region. Real materials, however, are anything but uniform. Electron densities peak sharply at nuclei, decay exponentially into vacuum, and exhibit complex variations in bonding regions. The **Generalized Gradient Approximation (GGA)** was developed to address this limitation by incorporating information about the **gradient** of the density.

### 5.2.1 The Central Idea

GGA retains the basic form of LDA but makes the exchange-correlation energy per electron depend not only on the local density $n(\mathbf{r})$ but also on the **gradient of the density**, $\nabla n(\mathbf{r})$:

$$E_{\text{xc}}^{\text{GGA}}[n] = \int n(\mathbf{r}) \, \varepsilon_{\text{xc}}^{\text{GGA}}(n(\mathbf{r}), \nabla n(\mathbf{r})) \, d\mathbf{r}$$

More sophisticated GGAs may also depend on higher-order derivatives or other local quantities, but the gradient is the essential new ingredient. The inclusion of gradient information allows GGA functionals to "sense" whether the density is rapidly varying (as near a nucleus) or slowly varying (as in simple metals) and adjust the exchange-correlation energy accordingly.

Typically, GGA functionals are written by separating exchange and correlation contributions:

$$E_{\text{xc}}^{\text{GGA}} = E_{\text{x}}^{\text{GGA}} + E_{\text{c}}^{\text{GGA}}$$

and are often expressed in terms of the **reduced density gradient**, a dimensionless quantity that measures how rapidly the density varies on the scale of the local Fermi wavelength:

$$s(\mathbf{r}) = \frac{|\nabla n(\mathbf{r})|}{2k_{\text{F}}(\mathbf{r}) n(\mathbf{r})} = \frac{|\nabla n(\mathbf{r})|}{2(3\pi^2)^{1/3} n^{4/3}(\mathbf{r})}$$

where $k_{\text{F}}(\mathbf{r}) = (3\pi^2 n(\mathbf{r}))^{1/3}$ is the local Fermi wavevector. When $s$ is small, the density is slowly varying and LDA should be reliable; when $s$ is large, the density is changing rapidly and gradient corrections become important.

### 5.2.2 Building GGA: Two Philosophical Approaches

GGA functionals are not unique—many different constructions exist that all depend on $n$ and $\nabla n$ but yield different numerical results. These constructions generally follow one of two philosophies:

**1. Empirical fitting:** Some GGAs are constructed by fitting parameters to experimental data sets (e.g., atomization energies, ionization potentials, bond lengths). The most famous example is the **BLYP functional** (Becke 1988 exchange + Lee-Yang-Parr 1988 correlation), which combines Becke's gradient-corrected exchange with the LYP correlation functional. These functionals can achieve high accuracy for molecular properties but may sacrifice physical constraints.

**2. Constraint satisfaction:** Another approach, championed by John Perdew and colleagues, builds functionals by enforcing exact physical constraints that the true exchange-correlation functional must satisfy. The philosophy is that a functional satisfying more exact constraints will be more universally reliable. The **PBE functional** (Perdew-Burke-Ernzerhof, 1996) is the most prominent example of this approach, designed to satisfy as many exact conditions as possible with no empirical fitting to experimental data.

### 5.2.3 The PBE Functional

PBE has become the most widely used GGA functional in condensed matter physics and materials science due to its simplicity, numerical stability, and consistently reasonable performance across diverse systems.

The PBE exchange energy takes the form:

$$E_{\text{x}}^{\text{PBE}}[n] = \int n(\mathbf{r}) \, \varepsilon_{\text{x}}^{\text{unif}}(n(\mathbf{r})) \, F_{\text{x}}^{\text{PBE}}(s(\mathbf{r})) \, d\mathbf{r}$$

where $\varepsilon_{\text{x}}^{\text{unif}}$ is the LDA exchange energy per electron, and $F_{\text{x}}^{\text{PBE}}(s)$ is an **enhancement factor** that modifies the LDA result based on the reduced density gradient:

$$F_{\text{x}}^{\text{PBE}}(s) = 1 + \kappa - \frac{\kappa}{1 + \mu s^2 / \kappa}$$

Here, $\kappa = 0.804$ and $\mu \approx 0.2195$ are constants chosen to satisfy exact constraints. For small $s$ (slowly varying density), $F_{\text{x}} \approx 1 + \mu s^2$, recovering the correct gradient expansion. For large $s$, $F_{\text{x}}$ saturates at $1 + \kappa$, preventing unphysical behavior.

The PBE correlation energy is constructed similarly, ensuring the total functional satisfies constraints like the uniform density limit, the Lieb-Oxford bound, and correct behavior under scaling transformations.

### 5.2.4 Key GGA Functionals

Several GGA functionals are commonly encountered in the literature:

| Functional | Type | Description |
|------------|------|-------------|
| **PBE** (1996) | Non-empirical | The workhorse of materials science; satisfies many exact constraints |
| **BLYP** (1988) | Empirical | Becke exchange + Lee-Yang-Parr correlation; popular in quantum chemistry |
| **PW91** (1991) | Non-empirical | Perdew-Wang 1991; predecessor to PBE, still widely used |
| **RPBE** (1999) | Non-empirical | Revised PBE; improves adsorption energies on surfaces |
| **PBEsol** (2008) | Non-empirical | Revised PBE for better equilibrium properties of solids and surfaces |
| **AM05** (2005) | Non-empirical | Built from the Airy gas model; excellent for surface properties |

### 5.2.5 Strengths and Improvements Over LDA

GGA corrects many of LDA's most serious deficiencies:

- **Improved binding energies:** GGA significantly reduces LDA's overbinding, giving atomization energies typically within 5-10% of experiment
- **Better bond lengths and angles:** Molecular geometries are improved, with bond lengths typically within 1-2% of experiment
- **More accurate energy barriers:** Reaction barriers, while still underestimated, are better than LDA
- **Magnetic ground states:** GGA often correctly predicts magnetic ordering where LDA fails
- **Hydrogen bonding:** GGA provides a reasonable description of hydrogen bonds, enabling studies of water and biological systems

### 5.2.6 Persistent Limitations

Despite significant improvements, GGA retains important limitations inherited from its semi-local nature:

- **Band gap problem:** GGA still dramatically underestimates semiconductor and insulator band gaps (typically by 30-50%)
- **No van der Waals interactions:** Standard GGAs, like LDA, fail to capture long-range dispersion forces, though some GGAs (like vdW-DF) are specially constructed to include them
- **Self-interaction error reduced but not eliminated:** GGA mitigates but does not cure the self-interaction problem, leading to over-delocalization of electrons and too-low reaction barriers
- **Strong correlation:** GGA fails for strongly correlated systems (transition metal oxides, rare earth compounds) where electrons are highly localized
- **Charge transfer excitations:** GGA poorly describes excited states involving electron transfer between distant sites
- **Surface adsorption energies:** While improved over LDA, GGAs like PBE still show systematic errors for molecule-surface interactions

### 5.2.7 Comparing GGA and LDA: The Paradigm Shift

The transition from LDA to GGA represented a paradigm shift in DFT practice:

| Property | LDA | GGA (PBE) | Experiment |
|----------|-----|-----------|------------|
| Si lattice constant [Å] | 5.40 | 5.47 | 5.43 |
| Si bulk modulus [GPa] | 98 | 89 | 98 |
| CO bond length [Å] | 1.13 | 1.14 | 1.13 |
| CO atomization energy [eV] | -12.3 | -11.2 | -11.2 |
| H₂ bond length [Å] | 0.76 | 0.75 | 0.74 |
| H₂ binding energy [eV] | -4.8 | -4.6 | -4.75 |

The pattern is clear: LDA tends to overbind (too short bonds, too large binding energies), while GGA often corrects this tendency, bringing results closer to experiment. This improvement is particularly dramatic for molecular properties and systems with significant density inhomogeneity.

### 5.2.8 When to Choose Which GGA

The choice of GGA functional depends on the system and properties of interest:

- **PBE:** The default choice for solids, surfaces, and general materials science applications
- **BLYP:** Often preferred for molecular chemistry, particularly organic molecules
- **RPBE:** Recommended for adsorption energies on metal surfaces (e.g., catalysis studies)
- **PBEsol:** Better lattice constants and surface energies for densely-packed solids
- **AM05:** Excellent for surface properties and work functions

### 5.2.9 The Next Step: Beyond GGA

GGA functionals represent the second rung on Jacob's Ladder and remain the workhorse of most DFT calculations today. However, their limitations—particularly the band gap problem, missing van der Waals interactions, and self-interaction error—motivate the next rungs: **meta-GGAs**, which introduce dependence on the kinetic energy density, and **hybrid functionals**, which incorporate exact Fock exchange from Hartree-Fock theory. These more advanced approximations recover additional exact constraints and improve accuracy for specific classes of problems, though at increased computational cost.

### 5.2.10 Key Takeaway

GGA functionals, particularly PBE, represent the most widely used approximation in modern DFT practice. By incorporating density gradient information, they correct LDA's overbinding and provide qualitatively correct descriptions of bonding, magnetism, and hydrogen bonding across a vast range of materials. While not the final word, GGAs strike an excellent balance between accuracy and computational efficiency, making them the default choice for most routine calculations.

## 6. Meta-GGA

**Key idea:** Adds dependence on the kinetic energy density (second derivatives of the orbitals), providing higher accuracy for systems with strong electron correlation .

**Characteristics** :
- ✅ Higher accuracy in systems with strong electron correlation
- ❌ Still suffers from self-interaction error
- ❌ More computationally demanding than GGA

### 6.1 Hybrid Functionals

**Key idea:** Partially incorporates **exact Fock exchange** from Hartree-Fock theory with semi-local exchange-correlation from DFT .

**Characteristics** :
- ✅ Accurately predict band structures in various systems
- ✅ Improved description of molecular properties
- ❌ High computational resources and costs needed
- ❌ Not suitable for all material types (e.g., metals can be problematic)

### 6.2 Advanced Corrections

**Hubbard U correction (DFT+U):** Integrates cost-effective Hubbard-like models into LDA or GGA .
- ✅ Adept at handling systems with **strong electronic correlations** (transition metals, rare earths)
- ❌ Difficult to accurately determine Hubbard U parameters

**van der Waals corrections:** Essential for layered materials, molecular crystals, and biological systems .
- **DFT-D2:** Fixed C6 coefficients
- **DFT-D3:** Refined C6 coefficients based on atomic coordination numbers
- **Tkatchenko-Scheffler:** Environment-dependent coefficients from electronic density


# 7. Current Frontiers

## 7.1 Machine Learning for Functional Development

Recent advances use machine learning to construct exchange-correlation functionals directly from data . This approach can be viewed as a sophisticated extension of traditional parameter fitting, offering new practical tools and insights into the structure of DFT. The challenge is finding formulas that relate the density $n(\mathbf{r})$ and exchange-correlation energy $E_{\text{xc}}$, which are both outputs of the Schrödinger equation—effectively approximating the sequential inverse-and-forward solution of the Schrödinger equation .

## 7.2 Ongoing Challenges

- **Band gap problem:** Standard functionals systematically underestimate band gaps 
- **Strong correlation:** Systems with localized d or f electrons remain challenging 
- **van der Waals interactions:** Essential for many biological and layered materials 
- **Self-interaction error:** Electrons improperly interact with themselves in semi-local approximations 

# 8. Summary and Key Takeaways

1. Exchange-correlation functionals **contain all many-body effects** not captured by the Hartree term
2. The exact functional is unknown—**all practical calculations use approximations**
3. Functionals exist on a ladder of complexity: LDA → GGA → meta-GGA → Hybrid → ... 
4. **No single functional works for all systems**—choice depends on the physics and chemistry of interest
5. Modern developments include **DFT+U for correlated systems, van der Waals corrections, and machine-learned functionals**

# 9. Recommended Reading

- Richard M. Martin, *Electronic Structure: Basic Theory and Practical Methods* (Cambridge University Press, 2004) 
- Tsuneda, *Density Functional Theory in Quantum Chemistry* (Springer, 2014) 
